### Import libraries

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.preprocessing import StandardScaler

### Load game and extra-statistics data

In [2]:
data_dir = Path("data") if Path("data").is_dir() else Path("../data")

ids = {"game_id": "string", "season_id": "string", "team_id_home": "string", "team_id_away": "string"}
games = pd.read_csv(data_dir / "game.csv", dtype=ids, parse_dates=["game_date"])
# other_stats has 10 exact duplicate rows (all outside the selected seasons)
stats = pd.read_csv(data_dir / "other_stats.csv", dtype=ids).drop_duplicates()

### Keep regular-season games from 2010–11 onward that have extra statistics

In [3]:
season_year = games["season_id"].str[-4:].astype(int)
games = games[
    (games["season_type"] == "Regular Season")
    & (season_year >= 2010)
    & games["game_id"].isin(stats["game_id"])
]
print("Games:", len(games))

Games: 12074


### One row per team per game

In [4]:
def stack_sides(df, cols, keep=("game_id",)):
    """Turn *_home / *_away column pairs into one row per team."""
    keep = list(keep)
    return pd.concat(
        df[keep + [f"{c}_{side}" for c in cols]].set_axis(keep + cols, axis=1).assign(side=side)
        for side in ("home", "away")
    )


box = stack_sides(
    games,
    ["team_id", "wl", "pts", "fgm", "fga", "fg3m", "fg3a", "reb", "tov"],
    keep=("game_id", "game_date", "season_id"),
)
extra = stack_sides(stats, ["team_id", "pts_paint", "pts_2nd_chance", "pts_fb", "pts_off_to"]).drop(columns="side")

# other_stats has home/away swapped in about half its games, so match on team_id, not side.
team_games = box.merge(extra, on=["game_id", "team_id"], validate="one_to_one")
team_games["win"] = (team_games["wl"] == "W").astype(float)
team_games["pts_allowed"] = team_games.groupby("game_id")["pts"].transform("sum") - team_games["pts"]

### Five-game rolling means (previous games only, same team and season)

In [5]:
WINDOW = 5
features = {
    "win_rate": "win",
    "points_scored_mean": "pts",
    "points_allowed_mean": "pts_allowed",
    "rebounds_mean": "reb",
    "turnovers_mean": "tov",
    "paint_points_mean": "pts_paint",
    "second_chance_points_mean": "pts_2nd_chance",
    "fast_break_points_mean": "pts_fb",
    "points_off_turnovers_mean": "pts_off_to",
}
source_cols = list(features.values()) + ["fgm", "fga", "fg3m", "fg3a"]

team_games = team_games.sort_values(["game_date", "game_id"])
rolled = team_games.groupby(["team_id", "season_id"])[source_cols].transform(
    # shift(1) first so the current game never feeds its own predictors
    lambda s: s.shift(1).rolling(WINDOW).mean()
)

history = team_games[["game_id", "side"]].copy()
for name, col in features.items():
    history[name] = rolled[col]
# Ratio of 5-game means equals ratio of 5-game totals
history["fg_pct"] = rolled["fgm"] / rolled["fga"]
history["fg3_pct"] = rolled["fg3m"] / rolled["fg3a"]

### One row per game; require five prior games for both teams

In [6]:
stems = [*features, "fg_pct", "fg3_pct"]
feature_cols = [f"{side}_{stem}_last{WINDOW}" for side in ("home", "away") for stem in stems]

wide = history.pivot(index="game_id", columns="side").dropna()
wide.columns = [f"{side}_{stem}_last{WINDOW}" for stem, side in wide.columns]

### Add labels and build the final dataset

In [7]:
games = games.set_index("game_id")
labels = pd.DataFrame({
    "home_win": (games["wl_home"] == "W").astype(int),
    "point_differential": games["pts_home"] - games["pts_away"],
    "pts_home": games["pts_home"],
    "pts_away": games["pts_away"],
})

df = (
    games[["game_date", "season_id", "team_id_home", "team_id_away"]]
    .join(wide[feature_cols], how="inner")
    .join(labels)
    .sort_values(["game_date", "game_id"])
)
assert len(df) == 11_086, len(df)

### Scale features

In [8]:
df[feature_cols] = StandardScaler().fit_transform(df[feature_cols])

print(df.shape)
df.head()

(11086, 30)


,game_date,season_id,team_id_home,team_id_away,home_win_rate_last5,home_points_scored_mean_last5,home_points_allowed_mean_last5,home_rebounds_mean_last5,home_turnovers_mean_last5,home_paint_points_mean_last5,...,away_paint_points_mean_last5,away_second_chance_points_mean_last5,away_fast_break_points_mean_last5,away_points_off_turnovers_mean_last5,away_fg_pct_last5,away_fg3_pct_last5,home_win,point_differential,pts_home,pts_away
game_id,,,,,,,,,,,,,,,,,,,,,
0021000074,2010-11-05,22010,1610612750,1610612737,-1.147144,-1.032638,0.882958,1.559222,2.775485,-1.070859,...,-0.407444,-0.749957,1.678800,-0.926960,0.852460,-0.693297,0,-10.0,103.0,113.0
0021000083,2010-11-06,22010,1610612749,1610612740,-0.378313,-1.694900,-1.380728,-0.632014,1.193575,-1.694033,...,-1.216227,-1.113933,0.245612,-1.125181,0.516931,0.101318,0,-6.0,81.0,87.0
0021000088,2010-11-06,22010,1610612758,1610612763,0.390519,0.132028,0.379917,-0.519643,-1.179288,-0.011462,...,0.774623,0.050791,2.395394,0.592740,0.132297,1.241628,0,-9.0,91.0,100.0
0021000089,2010-11-07,22010,1610612752,1610612755,0.390519,-0.073501,-0.649031,-0.013973,0.798098,-0.073779,...,-0.034160,-0.240390,2.293024,-0.266221,0.067302,1.092892,0,-10.0,96.0,106.0
0021000090,2010-11-07,22010,1610612737,1610612756,1.928182,-0.438887,-1.312131,-0.407272,0.204882,-0.634636,...,0.463552,1.215515,1.218132,1.385627,-0.087895,0.964255,0,-4.0,114.0,118.0
